<a href="https://colab.research.google.com/github/EirikEspe/10-Days-of-Statistics/blob/master/telia-ddi-no/no_82858_ducky_km_travelled_al2_purpose_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro

Ticket: https://dev.azure.com/teliaiot/DivX%20Data%20Insights/_boards/board/t/Delivery%20Kanban/Stories/?workitem=82858

Predecessor ticket: https://dev.azure.com/teliaiot/DivX%20Data%20Insights/_boards/board/t/Delivery%20Kanban/Stories/?workitem=64669

V1 delivery: https://github.com/ddi-gcp-admin/ddi-delivery/blob/master/telia-ddi-no/no_64669_ducky_km_travelled.ipynb

V1 delivery (2021 data): https://github.com/ddi-gcp-admin/ddi-delivery/blob/master/telia-ddi-no/no_64669_ducky_km_travelled_2021.ipynb

In [2]:
PROJECT_ID = 'telia-ddi-delivery'
DATASET_ID = 'no_82858_Ducky_Norway_trips_km'

# Connect, authenticate and set-up resources

In [3]:
from google.colab import auth
from google.cloud import storage
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()
client = bigquery.Client(PROJECT_ID)

# V2 Aggregate


Fetching blueprint query from: https://542471cf561345c5bb0784124e8fef64-dot-europe-west1.composer.googleusercontent.com/rendered-templates?dag_id=routed-odm_v2_prod_no&task_id=rodm_daily_oal4v2020_dal4v2020_db_odm_admin_level&execution_date=2023-07-03T12%3A03%3A30.833431%2B00%3A00


In [ ]:
%%bigquery --project telia-ddi-delivery
-- CREATE OR REPLACE TABLE telia-ddi-delivery.no_82858_Ducky_Norway_trips_km.rodm_aoh_purpose_v2 PARTITION BY batch_date AS
WITH transits AS (
  SELECT
    t.batch_date,
    t.identifier,
    t.transit_id,
    mb.admin_level_2_code AS multiplier_area_code,
    mo.admin_level_2_code AS origin_area_code,
    md.admin_level_2_code AS destination_area_code,
    aoh_al.admin_level_2_code AS home_area_code,
    CASE
      WHEN oaoh.aoh_classification = 'WORK' THEN 'WORK'
      WHEN daoh.aoh_classification = 'WORK' THEN 'WORK'
      WHEN oaoh.aoh_classification IS NULL THEN 'OTHER'
      WHEN daoh.aoh_classification IS NULL THEN 'OTHER'
      WHEN oaoh.aoh_classification = 'HOME' THEN 'OTHER'
      WHEN daoh.aoh_classification = 'HOME' THEN 'OTHER'
      ELSE oaoh.aoh_classification
    END as purpose,
    t.duration,
    t.air_distance,
    t.air_speed,
    t.distance,
    t.speed,
  FROM `telia-ddi-no-prod.uc_v2_mobility_signals.routed_transits_db` t
  INNER JOIN `telia-ddi-no-prod.uc_v2_maps.administrative_level_4_2020` mo ON ST_CONTAINS(mo.geo, t.origin_point)
  INNER JOIN `telia-ddi-no-prod.uc_v2_maps.administrative_level_4_2020` md ON ST_CONTAINS(md.geo, t.destination_point)
  # Join with base map for multiplier area_code
  INNER JOIN `telia-ddi-no-prod.uc_v2_maps.administrative_level_4` mb ON ST_CONTAINS(mb.geo, t.origin_point)

  JOIN `telia-ddi-no-prod.uc_v2_mobility_signals.areas_of_habit` aoh USING(identifier, batch_date)
  JOIN `telia-ddi-no-prod.uc_v2_maps.administrative_level_2_2020` aoh_al ON ST_INTERSECTS(aoh.point, aoh_al.geo)
  JOIN `telia-ddi-no-prod.uc_v2_mobility_signals.dwells` oaoh ON (t.origin_activity_id = oaoh.activity_id AND t.batch_date = oaoh.batch_date)
  JOIN `telia-ddi-no-prod.uc_v2_mobility_signals.dwells` daoh ON (t.destination_activity_id = daoh.activity_id AND t.batch_date = daoh.batch_date)

  WHERE t.batch_date BETWEEN '2022-01-01' AND '2023-11-30'
)

, transits_mot_rank AS (
  SELECT
    batch_date,
    transit_id,
    M.type as mode_of_transport,
    CASE
      WHEN M.type = "RAIL" AND M.distance / T.distance > 0.5 THEN 0
      ELSE ROW_NUMBER() OVER(PARTITION BY transit_id, batch_date ORDER BY M.distance DESC)
    END AS mode_of_transport_rank
  FROM `telia-ddi-no-prod.uc_v2_mobility_signals.routed_transits_db` T, UNNEST(mode_of_transport) M
  WHERE M.type IS NOT NULL -- Happens when we create aggregates on non-routed ODMs
    AND T.batch_date BETWEEN '2022-01-01' AND '2023-11-30'
)

, select_mot AS (
  SELECT
    batch_date,
    transit_id,
    MIN(mode_of_transport_rank) as min_rank
  FROM transits_mot_rank
  GROUP BY
    batch_date,
    transit_id
)

, transits_mot AS (
  SELECT
    T.batch_date,
    identifier,
    multiplier_area_code,
    origin_area_code,
    destination_area_code,
    home_area_code,
    purpose,
    -- Remove this case statement if you want to distinguish between Noise, Too short and Unspecified
    CASE
      WHEN mode_of_transport IN ("ROAD", "RAIL", "WALKING", "FERRY", "AIRPLANE") THEN mode_of_transport
      ELSE "UNKNOWN"
    END as mode_of_transport,
    duration,
    air_distance,
    air_speed,
    distance,
    speed,
    PERCENTILE_CONT(duration, 0.5) OVER odm_hour AS duration_median,
    PERCENTILE_CONT(air_distance, 0.5) OVER odm_hour AS air_distance_median,
    PERCENTILE_CONT(air_speed, 0.5) OVER odm_hour AS air_speed_median,
    PERCENTILE_CONT(distance, 0.5) OVER odm_hour AS distance_median,
    PERCENTILE_CONT(speed, 0.5) OVER odm_hour AS speed_median
  FROM transits T
  INNER JOIN (
    SELECT
      A.batch_date,
      A.transit_id,
      mode_of_transport
    FROM transits_mot_rank A
    INNER JOIN select_mot B
      ON A.transit_id = B.transit_id AND A.mode_of_transport_rank = B.min_rank AND A.batch_date = B.batch_date
  ) M ON T.transit_id = M.transit_id AND T.batch_date = M.batch_date
  WINDOW odm_hour AS (PARTITION BY  origin_area_code, destination_area_code, home_area_code, purpose, mode_of_transport)
)

, transits_total AS (
SELECT
  batch_date,
  identifier,
  multiplier_area_code,
  origin_area_code,
  destination_area_code,
  home_area_code,
  purpose,
  "TOTAL" AS mode_of_transport,
  duration,
  air_distance,
  air_speed,
  distance,
  speed,
  PERCENTILE_CONT(duration, 0.5) OVER odm_hour AS duration_median,
  PERCENTILE_CONT(air_distance, 0.5) OVER odm_hour AS air_distance_median,
  PERCENTILE_CONT(air_speed, 0.5) OVER odm_hour AS air_speed_median,
  PERCENTILE_CONT(distance, 0.5) OVER odm_hour AS distance_median,
  PERCENTILE_CONT(speed, 0.5) OVER odm_hour AS speed_median
FROM transits
WINDOW odm_hour AS (PARTITION BY origin_area_code, destination_area_code, home_area_code, purpose, batch_date)
)

, unioned_transits AS (
  SELECT * FROM transits_mot
  UNION ALL
  SELECT * FROM transits_total
)

, transits_avg AS (
  SELECT
    unioned_transits.batch_date,
    origin_area_code,
    destination_area_code,
    home_area_code,
    purpose,
    mode_of_transport,
    -- I didn't manage to solve the multiplier for higher levels + grids in a smooth way so now it is some crazy averages of averages
    AVG(multiplier) AS multiplier,
    AVG(duration) AS duration_avg,
    ANY_VALUE(duration_median) AS duration_median,
    AVG(air_distance) AS air_distance_avg,
    ANY_VALUE(air_distance_median) AS air_distance_median,
    AVG(air_speed) AS air_speed_avg,
    ANY_VALUE(air_speed_median) AS air_speed_median,
    AVG(distance) AS distance_avg,
    ANY_VALUE(distance_median) AS distance_median,
    AVG(speed) AS speed_avg,
    ANY_VALUE(speed_median) AS speed_median
  FROM unioned_transits
  LEFT JOIN
    (SELECT
        admin_level_2_code,
        batch_date,
        multiplier AS multiplier,
      FROM `telia-ddi-no-prod.uc_v2_extrapolation.multipliers_daily`
      WHERE aoh_classification = "TOTAL"
      ) mp
    ON multiplier_area_code = mp.admin_level_2_code
    AND mp.batch_date = unioned_transits.batch_date
  GROUP BY
    batch_date,
    origin_area_code,
    destination_area_code,
    home_area_code,
    purpose,
    mode_of_transport
)

, signal_probability AS (
  SELECT
    batch_date,
    (1 / p_observation) AS m_customers,
  FROM `telia-ddi-no-prod.uc_v2_extrapolation.signal_probability_day`
  WHERE batch_date BETWEEN '2022-01-01' AND '2023-11-30'
)

, identifiers_to_keep AS (
    SELECT
        batch_date,
        identifier,
        CASE
            WHEN observed_market_share IS NOT NULL THEN observed_market_share
            ELSE PERCENTILE_CONT(observed_market_share, 0.5) OVER()
            END AS observed_market_share
    FROM `telia-ddi-no-prod.uc_v2_mobility_signals.identifiers`
    WHERE batch_date BETWEEN '2022-01-01' AND '2023-11-30'
    AND has_home
    AND NOT (IFNULL(is_roaming, FALSE) AND batch_date >= '2020-09-28')
)

, ex_transits_num AS (
  SELECT
    batch_date,
    origin_area_code,
    destination_area_code,
    home_area_code,
    purpose,
    mode_of_transport,
    SUM(CASE WHEN identifier IS NOT NULL THEN 1 ELSE 0 END) AS devices,
    SUM(CASE WHEN identifier IS NOT NULL THEN m_customers * multiplier ELSE 0 END) AS customers_corrected,
    SUM(CASE WHEN identifier IS NOT NULL THEN (1/observed_market_share) * m_customers * multiplier ELSE 0 END) AS people_corrected,
    SUM(CASE WHEN identifier IS NOT NULL THEN m_customers ELSE 0 END) AS customers_uncorrected,
    SUM(CASE WHEN identifier IS NOT NULL THEN (1/observed_market_share) * m_customers ELSE 0 END) AS people_uncorrected
  FROM unioned_transits t
  JOIN signal_probability USING(batch_date)
  INNER JOIN identifiers_to_keep i USING(identifier, batch_date)
  LEFT JOIN transits_avg USING( origin_area_code, destination_area_code, home_area_code, purpose, mode_of_transport, batch_date)
  GROUP BY
    batch_date,
    origin_area_code,
    destination_area_code,
    home_area_code,
    purpose,
    mode_of_transport
  --HAVING
    --SUM((1/observed_market_share) * m_customers_morning) >= 0
)
-- #Calculate how many days of history available to calculate impute basis. We will only impute if we have 2 or more days
-- , days_with_good_history as (
-- SELECT
--     ARRAY_LENGTH(SPLIT('"2023-07-02", "2023-06-11", "2023-06-04", "2023-05-21", "2023-05-14"', ',')) num_days_history
-- )
-- , impute_basis as (
-- SELECT
--   c.batch_date,

--   origin_area_code,
--   destination_area_code,
--   mode_of_transport,
--   people,
--   ROW_NUMBER() OVER(PARTITION BY  origin_area_code, destination_area_code, mode_of_transport
--     ORDER BY c.batch_date DESC) AS rnk
-- FROM
-- `telia-ddi-no-prod.uc_v2_mobility_insights.rodm_daily_oal4v2020_dal4v2020_db` c
-- WHERE c.batch_date IN ("2023-07-02", "2023-06-11", "2023-06-04", "2023-05-21", "2023-05-14")
-- )
-- , impute_average AS (
-- SELECT

--   origin_area_code,
--   destination_area_code,
--   mode_of_transport,
--   AVG(people) as people_imputed
-- FROM impute_basis
-- WHERE batch_date != PARSE_DATE("%Y%m%d", '20230702')
-- GROUP BY

--   origin_area_code,
--   destination_area_code,
--   mode_of_transport
-- )

SELECT
  n.batch_date,
  n.origin_area_code,
  n.destination_area_code,
  n.home_area_code,
  n.purpose,
  n.mode_of_transport,
  CAST(devices AS INT64) AS devices,
  CASE WHEN NOT(is_detected) OR is_holiday OR is_summer THEN CAST(customers_uncorrected AS INT64)
       ELSE CAST(customers_corrected AS INT64)
       END AS customers,
  -- People contains the number that should be used, corrected or imputed for bad supply days and uncorrected for good days
    -- CASE
    --     WHEN NOT(is_detected) OR is_holiday OR is_summer
    --     THEN CAST(people_uncorrected AS INT64)
        -- We want to impute if the filtered multiplier is very large
        -- (defined as a max z score of over 3 std from mean)
        -- If num_days_history < 3 then we have less than 2 good days
        -- of history to impute or correct and it is better to do nothing.
    --     WHEN is_impute AND is_detected AND num_days_history >= 3 THEN
    --     IFNULL(
    --         CAST(i.people_imputed AS INT64),
    --         CAST(people_uncorrected AS INT64))
    --     ELSE
    --     CASE
    --         -- If very few unfiltered devices, it is better to do nothing
    --         WHEN devices < 3 THEN CAST(people_uncorrected AS INT64)
    --         -- if num_days_history < 3 then we have less than 2 good days of
    --         -- history to impute or correct and it is better to do nothing.
    --         WHEN num_days_history <3 THEN CAST(people_uncorrected AS INT64)
    --         ELSE CAST(people_corrected AS INT64)
    --         END
    -- END AS people,
    CASE
        WHEN NOT(is_detected) OR is_holiday OR is_summer THEN CAST(people_uncorrected AS INT64)
        -- WHEN num_days_history  < 3 OR devices < 3 THEN CAST(people_uncorrected AS INT64)
        WHEN devices < 3 THEN CAST(people_uncorrected AS INT64)
        ELSE CAST(people_corrected AS INT64)
    END AS people,
    CAST(customers_uncorrected AS INT64) as customers_uncorrected,
    CAST(customers_corrected AS INT64) AS customers_corrected,
    CAST(people_uncorrected AS INT64) AS people_uncorrected,
    CAST(people_corrected AS INT64) AS people_corrected,
    -- CAST(i.people_imputed AS INT64) AS people_imputed,
    ROUND(s.duration_avg, 2) AS duration_avg,
    ROUND(s.duration_median, 2) AS duration_median,
    ROUND(s.air_distance_avg, 2) AS air_distance_avg,
    ROUND(s.air_distance_median, 2) AS air_distance_median,
    ROUND(s.air_speed_avg, 2) AS air_speed_avg,
    ROUND(s.air_speed_median, 2) AS air_speed_median,
    ROUND(s.distance_avg, 2) AS distance_avg,
    ROUND(s.distance_median, 2) AS distance_median,
    ROUND(s.speed_avg, 2) AS speed_avg,
    ROUND(s.speed_median, 2) AS speed_median
-- days_with_good_history is a constant that is used to decide if we have enough good history to impute or correct
FROM ex_transits_num N --, days_with_good_history
LEFT JOIN `telia-ddi-no-prod.uc_v2_extrapolation.v_supply_log` supply_log USING(batch_date)
-- LEFT JOIN impute_average i USING( origin_area_code, destination_area_code, mode_of_transport)
INNER JOIN transits_avg S USING( origin_area_code, destination_area_code, home_area_code, purpose, mode_of_transport, batch_date)
--end-sql
--)



--select * from agg
-- where origin_area_code = '3413'
-- and destination_area_code = '3411'
-- and home_area_code = '3403'
-- order by 1, 2, 3, 4, 5

-- -- only writing the info nessessary to the final table since the aggregate is too big to be kept in non-aggregated table
-- select
--   batch_date
--   , case when mode_of_transport in ('WALKING', 'ROAD') then 'OTHER' else mode_of_transport end mode_of_transport
--   , home_area_code as admin_level_2_code
--   , sum(people_corrected * distance_avg) / 1000 as distance
--   , sum(people_corrected) as people
-- from agg
-- group by 1, 2, 3

Query is running:   0%|          |

""


# V2 View

In [ ]:
%%bigquery --project telia-ddi-delivery
create or replace view telia-ddi-delivery.no_82858_Ducky_Norway_trips_km.distance_km_al2_aoh_purpose_v2 as
with agg as (
select
  format_date('%Y-%m', batch_date) year_month
  , mode_of_transport
  , purpose
  , home_area_code
  , sum(people * distance_avg) / 1000 as total_km
  , sum(people) as people
from `telia-ddi-delivery.no_82858_Ducky_Norway_trips_km.rodm_aoh_purpose_v2` rodm

join `telia-ddi-no-prod.uc_v2_quality_assurance.daily_quality_rating_al0` r
  on r.date = batch_date

where
  rodm.batch_date between '2022-01-01' and '2023-11-30'
  and rating in ('A', 'B', 'C')

group by
  year_month
  , mode_of_transport
  , purpose
  , home_area_code
)

select
  year_month
  , al2.admin_level_2 AS home_area
  , home_area_code
  , purpose
  , mode_of_transport
  , cast(total_km as int64) as total_km
  , cast(people as int64) as people
from agg
inner join `telia-ddi-no-prod.uc_v2_maps.administrative_level_2_2020` al2
  on agg.home_area_code = al2.admin_level_2_code

#where people >= 5

Query is running:   0%|          |

""


There were two days with signal rating D:


*   2022-06-23 (signal_quality: 68.78)
*   2023-03-29 (signal_quality: 69.19)



In [5]:
%%bigquery --project telia-ddi-delivery
SELECT
  date
  , rating
  , signal_quality
FROM `telia-ddi-no-prod.uc_v2_quality_assurance.daily_quality_rating_al0`
WHERE
  date BETWEEN '2022-01-01' AND '2023-11-30' AND
  rating NOT IN ('A', 'B', 'C')

Query is running:   0%|          |

Downloading:   0%|          |

,date,rating,signal_quality
0,2022-06-23,D,68.780397
1,2023-03-29,D,69.190683


Export data to bucket

In [ ]:
EXPORT DATA
  OPTIONS (
    uri = 'gs://no_64669_ducky/distance_travelled_2022_2023/kommune/distance_km_al2_aoh_purpose_v2_*.csv',
    format = 'CSV',
    overwrite = true,
    header = true,
    field_delimiter = ',')
AS (
  SELECT
  year_month
  , home_area
  , home_area_code
  , purpose
  , mode_of_transport
  , total_km
FROM `telia-ddi-delivery.no_82858_Ducky_Norway_trips_km.distance_km_al2_aoh_purpose_v2`
);